# NN Architecture 2B: 1D Convolutional Neural Network (CNN)

**Reference**: "Binary Case using Deep Learning" (project paper)

**Approach**: Learn local patterns in signal sequences (BPSK + TAG) through convolutional filters

**Rationale**: Convolutional layers with 1D kernels excel at:
- Detecting local correlations in signal sequences
- Learning shift-invariant features
- Reducing parameters vs. fully-connected layers
- Handling variable-length inputs (with padding)

**Architecture**:
```
Input: [τ_signal, h_magnitude, SNR_local, energy] (4D, reshaped as 1D sequence)
    ↓
Conv1D(32, kernel=5, padding='same', ReLU)
    ↓
BatchNorm → MaxPool(2, stride=2)
    ↓
Conv1D(64, kernel=3, padding='same', ReLU)
    ↓
BatchNorm → MaxPool(2, stride=2) → Flatten
    ↓
Dense(128, ReLU, Dropout=0.4)
    ↓
Dense(64, ReLU, Dropout=0.3)
    ↓
Dense(1, Sigmoid) → Binary output
```

**Framework**: PyTorch (better for signal processing + flexibility)

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATHS FOR NEW DIRECTORY STRUCTURE
# ============================================================================
from pathlib import Path

notebook_dir = Path.cwd()  # Current: notebooks/
project_root = notebook_dir.parent  # Go up to: Redes Neurais/
results_dir = project_root / "results"
data_dir = results_dir / "data"
models_dir = results_dir / "models"
visualizations_dir = results_dir / "visualizations"

models_dir.mkdir(parents=True, exist_ok=True)
visualizations_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / "dataset_nn_100k.h5"
with h5py.File(str(dataset_path), 'r') as f:
    X_train = f['X_train'][:]
    y_train = f['y_train'][:]
    X_val = f['X_val'][:]
    y_val = f['y_val'][:]
    X_test = f['X_test'][:]
    y_test = f['y_test'][:]

print(f"✓ Dataset loaded: X_train={X_train.shape}, y_train={y_train.shape}")


In [ ]:
# ==============================================================================
# 2. BUILD CNN ARCHITECTURE
# ==============================================================================

class CNN_SignalProcessing(nn.Module):
    """1D CNN for signal processing (Binary Case using Deep Learning)"""
    
    def __init__(self, input_channels=1, dropout_p=0.3):
        super(CNN_SignalProcessing, self).__init__()
        
        # Conv block 1
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2, stride=2)
        self.dropout1 = nn.Dropout(dropout_p)
        
        # Conv block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2, stride=2)
        self.dropout2 = nn.Dropout(dropout_p)
        
        # FC layers
        # After 2 maxpools, input_dim=4 → 4/4 ≈ 1, but we keep it larger for flexibility
        self.fc1 = nn.Linear(64 * 1, 128)  # 64 channels * reduced spatial dims
        self.dropout3 = nn.Dropout(0.4)
        
        self.fc2 = nn.Linear(128, 64)
        self.dropout4 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = torch.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = torch.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = torch.relu(self.fc1(x))
        x = self.dropout3(x)
        
        x = torch.relu(self.fc2(x))
        x = self.dropout4(x)
        
        x = self.fc3(x)
        x = self.sigmoid(x)
        
        return x

# Create model
model = CNN_SignalProcessing(input_channels=1, dropout_p=0.3)
model = model.to(device)

print("✓ CNN Model created:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

In [ ]:
        torch.save(model.state_dict(), str(models_dir / 'model_cnn_best.pth'))


In [ ]:
# ==============================================================================
# 4. TRAINING LOOP
# ==============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
    
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            all_preds.extend(y_pred.cpu().numpy().flatten())
            all_targets.extend(y_batch.cpu().numpy().flatten())
    
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_targets)

# Training
num_epochs = 100
patience = 10
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

print("Training CNN model...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, _, _ = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), str(models_dir / 'model_cnn_best.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("✓ Training completed!")

In [ ]:
output_file = visualizations_dir / 'results_cnn.png'
plt.savefig(str(output_file), dpi=100, bbox_inches='tight')
plt.show()
print(f"✓ Results saved to '{output_file.name}'")
